# Light-flavour Drell–Yan observable

This notebook compares a PineAPPL observable calculated using the `NNPDF_original` and `KDE_reconstruction` ensembles. It propagates the PDF uncertainty by calculating the observable with every Monte Carlo replica.

The reconstructed set contains $d$, $u$, $s$, $c$, their antiquarks and the gluon. The PineAPPL bottom and photon channels are excluded from both ensembles. 

Contributions outside the reconstruction x-grid are set to zero for both ensembles rather than extrapolated. 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from lhapdf_calc_observables import (
    ORIGINAL_SET_NAME,
    RECONSTRUCTED_SET_NAME,
    SCRIPT_DIR,
    ObservableStatistics,
    ObservablePlotStyle,
    PineAPPLObservable,
)

plot_style = ObservablePlotStyle()
plot_style.apply()
print(f"Using Matplotlib style: {plot_style.style_path}")

### Load and inspect the PineAPPL grid

Load the grid, create the two-proton convolution and select the supported light-flavour channels.

In [ ]:
observable = PineAPPLObservable()

observable.load()
print("Selected channels:", observable.supported_channels)
print("Bin limits:\n", observable.bin_limits())

### Calculate the central predictions

Calculate the 18 observable bins using member 0 of each PDF set before running any replicas.

In [ ]:
original_central = observable.calculate_member(ORIGINAL_SET_NAME, 0)
reconstructed_central = observable.calculate_member(RECONSTRUCTED_SET_NAME, 0)

print("Original central:\n", original_central)
print("Reconstructed central:\n", reconstructed_central)

### Test a small replica sample

Use 10 replicas from each ensemble to check the calculation and array shapes before running all 1,000 replicas. 

Replica indices start at 1 because member 0 is the central member.

In [ ]:
test_members = range(1, 11)

original_test = observable.calculate_ensemble(ORIGINAL_SET_NAME, test_members)
reconstructed_test = observable.calculate_ensemble(
    RECONSTRUCTED_SET_NAME, test_members
)

print("Original test shape:", original_test.shape)
print("Reconstructed test shape:", reconstructed_test.shape)

### Propagate the full PDF uncertainties

Calculate the observable for all 1,000 replicas in each ensemble. 

The expected shape of each result is `(1000, 18)`.

In [ ]:
all_replica_members = range(1, 1001)

original_replicas = observable.calculate_ensemble(
    ORIGINAL_SET_NAME, all_replica_members
)
reconstructed_replicas = observable.calculate_ensemble(
    RECONSTRUCTED_SET_NAME, all_replica_members
)

print("Original ensemble shape:", original_replicas.shape)
print("Reconstructed ensemble shape:", reconstructed_replicas.shape)

### Calculate ensemble statistics

Calculate the replica mean, sample standard deviation and central 68% percentile interval in every observable bin.

In [ ]:
original_statistics = ObservableStatistics(original_replicas).summary()
reconstructed_statistics = ObservableStatistics(
    reconstructed_replicas
).summary()

print("Original standard deviation:\n", original_statistics["standard_deviation"])
print("Reconstructed standard deviation:\n", reconstructed_statistics["standard_deviation"])

### Plot predictions with uncertainty 

Plot the ensemble means and uncertainty intervals (plus, minus one standard deviation) for the original and reconstructed sets.

In [ ]:
bin_centres = observable.bin_centres()

plot_style.apply()
fig, ax = plt.subplots()

original_line, = ax.plot(
    bin_centres, original_statistics["mean"],
    label=r"NNPDF original",
)
ax.fill_between(
    bin_centres,
    original_statistics["lower_68"],
    original_statistics["upper_68"],
    facecolor=original_line.get_color(), alpha=0.25,
)
reconstructed_line, = ax.plot(
    bin_centres, reconstructed_statistics["mean"],
    label=r"KDE reconstruction",
)
ax.fill_between(
    bin_centres,
    reconstructed_statistics["lower_68"],
    reconstructed_statistics["upper_68"],
    facecolor=reconstructed_line.get_color(), alpha=0.25,
)
ax.set_xlabel(r"Muon direction, $\eta$")
ax.set_ylabel(r"Predicted cross section")
plot_style.add_legend(ax)
plot_style.save(fig, "prediction_uncertainties.png")
plt.show()